# 01 — Data: load, combine, check

Builds one match-level table from the raw football-data.co.uk files (`data/raw/E0_*.csv`).

Raw files are downloaded by `src/download_data.py` and never edited. All cleaning happens here, in code.

## 1. Load every season

In [1]:
from pathlib import Path

import pandas as pd

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

# The only columns we need (see PLAN.md, section 2)
COLUMNS = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "Referee",
           "HF", "AF", "HY", "AY", "HR", "AR"]

pd.set_option("display.width", 200)

In [2]:
seasons = []

for path in sorted(RAW_DIR.glob("E0_*.csv")):
    code = path.stem.replace("E0_", "")           # e.g. "1011"
    season_start = 2000 + int(code[:2])           # e.g. 2010

    one_season = pd.read_csv(path, usecols=COLUMNS)
    one_season["season_start"] = season_start
    one_season["season"] = f"{season_start}/{str(season_start + 1)[2:]}"   # e.g. "2010/11"

    print(f"{path.name}: {one_season.shape[0]} rows")
    seasons.append(one_season)

matches = pd.concat(seasons, ignore_index=True)
print("\nCombined shape:", matches.shape)
matches.head()

E0_1011.csv: 380 rows
E0_1112.csv: 380 rows
E0_1213.csv: 380 rows
E0_1314.csv: 380 rows
E0_1415.csv: 381 rows
E0_1516.csv: 380 rows
E0_1617.csv: 380 rows
E0_1718.csv: 380 rows
E0_1819.csv: 380 rows
E0_1920.csv: 380 rows
E0_2021.csv: 380 rows
E0_2122.csv: 380 rows
E0_2223.csv: 380 rows

Combined shape: (4941, 14)


,Date,HomeTeam,AwayTeam,FTHG,FTAG,Referee,HF,AF,HY,AY,HR,AR,season_start,season
0,14/08/10,Aston Villa,West Ham,3.0,0.0,M Dean,15.0,15.0,1.0,2.0,0.0,0.0,2010,2010/11
1,14/08/10,Blackburn,Everton,1.0,0.0,P Dowd,19.0,14.0,2.0,1.0,0.0,0.0,2010,2010/11
2,14/08/10,Bolton,Fulham,0.0,0.0,S Attwell,12.0,13.0,1.0,3.0,0.0,0.0,2010,2010/11
3,14/08/10,Chelsea,West Brom,6.0,0.0,M Clattenburg,10.0,10.0,1.0,0.0,0.0,0.0,2010,2010/11
4,14/08/10,Sunderland,Birmingham,2.0,2.0,A Taylor,13.0,10.0,3.0,3.0,1.0,0.0,2010,2010/11


## 2. Empty rows

A row where every column we use is empty is not a match (a trailing blank line in the CSV). We list them before dropping.

In [3]:
empty_rows = matches[matches[COLUMNS].isna().all(axis=1)]
print("Fully empty rows:", len(empty_rows))
empty_rows

Fully empty rows: 1


,Date,HomeTeam,AwayTeam,FTHG,FTAG,Referee,HF,AF,HY,AY,HR,AR,season_start,season
1900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014,2014/15


In [4]:
matches = matches.drop(index=empty_rows.index).reset_index(drop=True)
print("Shape after dropping empty rows:", matches.shape)
print("\nMissing values in key columns:")
print(matches[COLUMNS].isna().sum())

Shape after dropping empty rows: (4940, 14)

Missing values in key columns:
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
Referee     0
HF          0
AF          0
HY          0
AY          0
HR          0
AR          0
dtype: int64


## 3. Dates

Seasons use two formats: `14/08/10` (2-digit year) and `08/08/2015` (4-digit year). Both are day/month/year.

In [5]:
print("Length of the Date text, per season:")
print(matches.groupby("season")["Date"].apply(lambda d: sorted(d.str.len().unique())))

Length of the Date text, per season:
season
2010/11     [8]
2011/12     [8]
2012/13     [8]
2013/14     [8]
2014/15     [8]
2015/16    [10]
2016/17     [8]
2017/18    [10]
2018/19    [10]
2019/20    [10]
2020/21    [10]
2021/22    [10]
2022/23    [10]
Name: Date, dtype: object


In [6]:
def parse_date(text):
    """Parse 'dd/mm/yy' or 'dd/mm/yyyy'."""
    if len(text) == 8:
        return pd.to_datetime(text, format="%d/%m/%y")
    return pd.to_datetime(text, format="%d/%m/%Y")


matches["Date"] = matches["Date"].apply(parse_date)

# Sanity check: every season should run from August/September to May/July
matches.groupby("season")["Date"].agg(["min", "max", "count"])

,min,max,count
season,,,
2010/11,2010-08-14,2011-05-22,380
2011/12,2011-08-13,2012-05-13,380
2012/13,2012-08-18,2013-05-19,380
2013/14,2013-08-17,2014-05-11,380
2014/15,2014-08-16,2015-05-24,380
2015/16,2015-08-08,2016-05-17,380
2016/17,2016-08-13,2017-05-21,380
2017/18,2017-08-11,2018-05-13,380
2018/19,2018-08-10,2019-05-12,380


## 4. Integer columns

Goals, fouls and cards were read as decimals (because of the empty row). They are counts, so we store them as whole numbers.

In [7]:
COUNT_COLUMNS = ["FTHG", "FTAG", "HF", "AF", "HY", "AY", "HR", "AR"]
matches[COUNT_COLUMNS] = matches[COUNT_COLUMNS].astype(int)
matches.dtypes

Date            datetime64[us]
HomeTeam                   str
AwayTeam                   str
FTHG                     int64
FTAG                     int64
Referee                    str
HF                       int64
AF                       int64
HY                       int64
AY                       int64
HR                       int64
AR                       int64
season_start             int64
season                     str
dtype: object

## 5. Data checks

Anything suspicious is listed here and decided explicitly — never fixed silently.

### 5a. Duplicate matches

In [8]:
duplicates = matches[matches.duplicated(["season", "HomeTeam", "AwayTeam"], keep=False)]
print("Duplicate (season, home, away) rows:", len(duplicates))

Duplicate (season, home, away) rows: 0


### 5b. Referees

In [9]:
print("Number of distinct referee names:", matches["Referee"].nunique())
matches["Referee"].value_counts().sort_index()

Number of distinct referee names: 43


Referee
A Madley          69
A Marriner       306
A Moss             1
A Taylor         342
C Foy            109
C Kavanagh       119
C Pawson         227
D Bond             4
D Coote           90
D England         45
G Scott           99
H Webb           119
J Brooks          23
J Gillett         26
J Moss           274
J Smith            1
K Friend         259
K Stroud           3
L Mason          226
L Probert        137
M Atkinson       333
M Clattenburg    175
M Dean           339
M Halsey          60
M Jones          170
M Oliver         350
M Salisbury       18
N Swarbrick      132
O Langford         1
P Bankes          60
P Dowd           130
P Tierney        158
P Walton          42
R East            91
R Jones           48
R Madley          92
S Attwell        163
S Hooper          77
S Scott            1
T Bramall          8
T Harrington       9
T Robinson         3
l Mason            1
Name: count, dtype: int64

In [10]:
# Referees who appear in only a handful of matches — possible typos
referee_counts = matches["Referee"].value_counts()
rare_referees = referee_counts[referee_counts <= 2].index

SHOW = ["season", "Date", "HomeTeam", "AwayTeam", "Referee", "HF", "AF", "HY", "AY", "HR", "AR"]
matches[matches["Referee"].isin(rare_referees)][SHOW]

,season,Date,HomeTeam,AwayTeam,Referee,HF,AF,HY,AY,HR,AR
2654,2016/17,2017-05-21,Leicester,Bournemouth,l Mason,11,10,4,1,0,0
3438,2019/20,2019-08-18,Chelsea,Leicester,O Langford,9,14,1,0,0,0
3642,2019/20,2020-01-18,Man City,Crystal Palace,S Scott,9,7,1,2,0,0
4159,2020/21,2021-05-16,Everton,Sheffield United,A Moss,8,12,1,3,0,0
4920,2022/23,2023-05-20,Fulham,Crystal Palace,J Smith,13,12,3,0,0,0


**Referee name corrections.** Each one-match referee was checked against match reports:

| Raw name | Match | Actual referee | Source |
|---|---|---|---|
| `l Mason` | Leicester–Bournemouth, 2017-05-21 | Lee Mason (lowercase typo) | same initials as `L Mason` |
| `A Moss` | Everton–Sheffield United, 2021-05-16 | Jon Moss | [ToffeeWeb match report](https://www.toffeeweb.com/season/20-21/reports/SheffUtd(h).php) |
| `S Scott` | Man City–Crystal Palace, 2020-01-18 | Graham Scott | [GJ Football Archive](https://gjfootballarchive.com/2021/01/18/agueros-250th-and-251st-goals/) |
| `O Langford` | Chelsea–Leicester, 2019-08-18 | Oliver Langford — **correct**: stepped in when Graham Scott was stuck in traffic | [Wikipedia](https://en.wikipedia.org/wiki/Oliver_Langford) |
| `J Smith` | Fulham–Crystal Palace, 2023-05-20 | Josh Smith — **correct**: Premier League debut | [premierleague.com](https://premierleague.com/match/75273) |

In [11]:
REFEREE_FIXES = {
    "l Mason": "L Mason",
    "A Moss": "J Moss",
    "S Scott": "G Scott",
}

matches["Referee"] = matches["Referee"].replace(REFEREE_FIXES)

print("Distinct referees after fixes:", matches["Referee"].nunique())
matches["Referee"].value_counts().tail(8)

Distinct referees after fixes: 40


Referee
M Salisbury     18
T Harrington     9
T Bramall        8
D Bond           4
K Stroud         3
T Robinson       3
O Langford       1
J Smith          1
Name: count, dtype: int64

### 5c. Teams

In [12]:
teams = sorted(set(matches["HomeTeam"]) | set(matches["AwayTeam"]))
print(len(teams), "teams:")
print(teams)

39 teams:
['Arsenal', 'Aston Villa', 'Birmingham', 'Blackburn', 'Blackpool', 'Bolton', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Cardiff', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Huddersfield', 'Hull', 'Leeds', 'Leicester', 'Liverpool', 'Man City', 'Man United', 'Middlesbrough', 'Newcastle', 'Norwich', "Nott'm Forest", 'QPR', 'Reading', 'Sheffield United', 'Southampton', 'Stoke', 'Sunderland', 'Swansea', 'Tottenham', 'Watford', 'West Brom', 'West Ham', 'Wigan', 'Wolves']


### 5d. Fouls and cards

A Premier League team usually commits around 10 fouls per match. Very low counts (0–2) are suspicious and matter for `cpf_diff` (cards per foul), because we divide by fouls.

In [13]:
matches[COUNT_COLUMNS].describe().T[["min", "mean", "50%", "max"]]

,min,mean,50%,max
FTHG,0.0,1.540283,1.0,9.0
FTAG,0.0,1.215182,1.0,9.0
HF,0.0,10.531377,10.0,24.0
AF,1.0,10.877328,11.0,26.0
HY,0.0,1.518016,1.0,7.0
AY,0.0,1.747368,2.0,9.0
HR,0.0,0.056883,0.0,2.0
AR,0.0,0.074899,0.0,2.0


In [14]:
low_fouls = matches[(matches["HF"] <= 2) | (matches["AF"] <= 2)]
print("Matches where a team has 0-2 fouls:", len(low_fouls))
print("Matches where a team has 0 fouls:", ((matches["HF"] == 0) | (matches["AF"] == 0)).sum())
low_fouls[SHOW]

Matches where a team has 0-2 fouls: 28
Matches where a team has 0 fouls: 2


,season,Date,HomeTeam,AwayTeam,Referee,HF,AF,HY,AY,HR,AR
131,2010/11,2010-11-20,Birmingham,Chelsea,M Halsey,9,2,2,0,0,0
388,2011/12,2011-08-15,Man City,Swansea,M Dean,9,2,0,0,0,0
410,2011/12,2011-09-10,Bolton,Man United,A Marriner,14,1,2,0,0,0
810,2012/13,2012-09-29,Everton,Southampton,L Probert,2,3,0,0,0,0
813,2012/13,2012-09-29,Norwich,Liverpool,M Jones,5,2,1,0,0,0
819,2012/13,2012-10-06,Chelsea,Norwich,A Taylor,7,1,0,0,0,0
1051,2012/13,2013-03-16,Swansea,Arsenal,J Moss,2,7,1,1,0,0
1122,2012/13,2013-05-12,Fulham,Liverpool,M Halsey,2,10,0,1,0,0
1158,2013/14,2013-08-25,Cardiff,Man City,L Probert,2,9,0,0,0,0
1177,2013/14,2013-09-14,Tottenham,Norwich,L Mason,2,12,1,3,0,0


In [15]:
print("Low-foul matches per season:")
low_fouls["season"].value_counts().sort_index()

Low-foul matches per season:


season
2010/11    1
2011/12    2
2012/13    5
2013/14    2
2014/15    1
2015/16    1
2016/17    1
2017/18    1
2018/19    3
2019/20    2
2020/21    3
2021/22    6
Name: count, dtype: int64

**Decision on low foul counts.** These look like data-entry errors (spread evenly over seasons, not concentrated in the no-fans period).
We keep all matches for `card_diff`, but flag them with `low_fouls = 1` and exclude them from `cpf_diff` (cards per foul), where dividing by 0–2 fouls gives undefined or extreme values.
The threshold (≤ 2 fouls for either team) is fixed here, before looking at any ghost-vs-crowd results.

In [16]:
LOW_FOULS_THRESHOLD = 2

matches["low_fouls"] = ((matches["HF"] <= LOW_FOULS_THRESHOLD) |
                        (matches["AF"] <= LOW_FOULS_THRESHOLD)).astype(int)

matches["low_fouls"].value_counts()

low_fouls
0    4912
1      28
Name: count, dtype: int64

In [17]:
# More yellows than fouls for one team: possible (dissent, time-wasting) but worth a look
more_yellows_than_fouls = matches[(matches["HY"] > matches["HF"]) | (matches["AY"] > matches["AF"])]
more_yellows_than_fouls[SHOW]

,season,Date,HomeTeam,AwayTeam,Referee,HF,AF,HY,AY,HR,AR
3871,2020/21,2020-11-07,Crystal Palace,Leeds,C Kavanagh,9,1,0,2,0,0


In [18]:
# Highest card counts — should be recognisable heated matches
matches.sort_values("AY", ascending=False).head(5)[SHOW]

,season,Date,HomeTeam,AwayTeam,Referee,HF,AF,HY,AY,HR,AR
2255,2015/16,2016-05-02,Chelsea,Tottenham,M Clattenburg,9,20,3,9,0,0
818,2012/13,2012-10-01,QPR,West Ham,M Clattenburg,9,21,1,8,1,0
198,2010/11,2011-01-02,Chelsea,Aston Villa,L Mason,17,11,2,7,0,0
4704,2022/23,2022-11-13,Brighton,Aston Villa,C Kavanagh,14,11,2,7,0,0
2854,2017/18,2017-12-26,Man United,Burnley,M Atkinson,10,14,3,7,0,0


## 6. Ghost / partial classification

Sources and reasoning: `data/ghost_matches_sources.md`. The 35 matches with limited attendance are listed by hand in `data/partial_matches.csv`.

Rule: a match is **partial** if it is on that list; it is **ghost** if it was played between 2020-06-17 and 2021-05-23 and is not partial.

In [19]:
partial_list = pd.read_csv("../data/partial_matches.csv", parse_dates=["date"])
print(partial_list.shape)
partial_list.head()

(35, 6)


,date,HomeTeam,AwayTeam,attendance,source,confidence
0,2020-12-05,Chelsea,Leeds,2000,https://www.premierleague.com/match/58999,high
1,2020-12-05,West Ham,Man United,2000,https://www.premierleague.com/match/59005,high
2,2020-12-06,Liverpool,Wolves,2000,https://www.premierleague.com/match/59000,high
3,2020-12-06,Tottenham,Arsenal,2000,https://www.premierleague.com/match/59003,high
4,2020-12-07,Brighton,Southampton,2000,https://www.premierleague.com/match/58997,high


In [20]:
# Mark partial matches by joining on (date, home, away).
# indicator=True adds a column telling us whether each row found a partner.
matches = matches.merge(
    partial_list[["date", "HomeTeam", "AwayTeam", "attendance"]],
    left_on=["Date", "HomeTeam", "AwayTeam"],
    right_on=["date", "HomeTeam", "AwayTeam"],
    how="left",
    indicator=True,
).drop(columns="date")

matches["partial"] = (matches["_merge"] == "both").astype(int)
matches = matches.drop(columns="_merge")

# Every match from the list must be found exactly once — otherwise a date or team name is wrong
print("Partial matches found:", matches["partial"].sum(), "of", len(partial_list))
assert matches["partial"].sum() == len(partial_list)

Partial matches found: 35 of 35


In [21]:
GHOST_START = pd.Timestamp("2020-06-17")   # first match after the restart
GHOST_END = pd.Timestamp("2021-05-23")     # last match day of 2020/21

in_covid_window = matches["Date"].between(GHOST_START, GHOST_END)
matches["ghost"] = (in_covid_window & (matches["partial"] == 0)).astype(int)

def crowd_label(row):
    if row["ghost"] == 1:
        return "ghost"
    if row["partial"] == 1:
        return "partial"
    return "full"

matches["crowd"] = matches.apply(crowd_label, axis=1)

pd.crosstab(matches["season"], matches["crowd"], margins=True)

crowd,full,ghost,partial,All
season,,,,
2010/11,380,0,0,380
2011/12,380,0,0,380
2012/13,380,0,0,380
2013/14,380,0,0,380
2014/15,380,0,0,380
2015/16,380,0,0,380
2016/17,380,0,0,380
2017/18,380,0,0,380
2018/19,380,0,0,380


## 7. Analysis variables (PLAN.md, section 2)

- `card_diff` = AY − HY (positive = away team got more yellows)
- `cpf_diff` = AY/AF − HY/HF (yellow cards per foul); empty for `low_fouls` matches
- `foul_diff` = AF − HF
- `red_diff` = AR − HR
- `var` = 1 from 2019/20 (VAR introduced in the Premier League)
- `trend` = season number (2010/11 = 0, 2011/12 = 1, ...)

`elo_diff` comes later, in stage 3.

In [22]:
matches["card_diff"] = matches["AY"] - matches["HY"]
matches["foul_diff"] = matches["AF"] - matches["HF"]
matches["red_diff"] = matches["AR"] - matches["HR"]

# Cards per foul — only where both teams have a believable number of fouls
cpf = matches["AY"] / matches["AF"] - matches["HY"] / matches["HF"]
matches["cpf_diff"] = cpf.where(matches["low_fouls"] == 0)

matches["var"] = (matches["season_start"] >= 2019).astype(int)
matches["trend"] = matches["season_start"] - 2010

matches[["season", "HomeTeam", "AwayTeam", "crowd", "card_diff", "cpf_diff", "foul_diff", "red_diff", "var", "trend"]].sample(5, random_state=1)

,season,HomeTeam,AwayTeam,crowd,card_diff,cpf_diff,foul_diff,red_diff,var,trend
1584,2014/15,Sunderland,Stoke,full,1,-0.064706,7,0,0,4
2275,2015/16,Stoke,West Ham,full,-2,-0.291667,4,0,0,5
3183,2018/19,West Ham,Cardiff,full,1,0.100000,0,0,0,8
2263,2015/16,Liverpool,Watford,full,1,0.033333,5,0,0,5
756,2011/12,Swansea,Liverpool,full,0,0.000000,0,0,0,1


In [23]:
print("cpf_diff missing:", matches["cpf_diff"].isna().sum(), "(should equal number of low_fouls matches:", matches["low_fouls"].sum(), ")")
matches[["card_diff", "cpf_diff", "foul_diff", "red_diff"]].describe().T[["count", "mean", "min", "max"]]

cpf_diff missing: 28 (should equal number of low_fouls matches: 28 )


,count,mean,min,max
card_diff,4940.0,0.229352,-6.000000,7.000000
cpf_diff,4912.0,0.018745,-0.666667,0.857143
foul_diff,4940.0,0.345951,-17.000000,17.000000
red_diff,4940.0,0.018016,-2.000000,2.000000


## 8. Stage 1 checkpoint

From PLAN.md: one match-by-match table; number of matches in each category; number of referees; no missing values in key columns.

Seasons 2010/11–2016/17 stay in the file — they are needed to warm up ELO (stage 3) and for placebo tests (stage 7). The analysis sample is 2017/18–2022/23.

In [24]:
analysis = matches[matches["season_start"] >= 2017]

print("All seasons:      ", matches.shape[0], "matches")
print("Analysis sample:  ", analysis.shape[0], "matches (2017/18-2022/23)")
print()
print("Matches per crowd category (analysis sample):")
print(analysis["crowd"].value_counts())
print()
print("Referees in analysis sample:", analysis["Referee"].nunique())
print("Referees with at least one ghost match:", analysis.loc[analysis["ghost"] == 1, "Referee"].nunique())
print()
print("Missing values in key columns:", matches[COLUMNS].isna().sum().sum())

All seasons:       4940 matches
Analysis sample:   2280 matches (2017/18-2022/23)

Matches per crowd category (analysis sample):
crowd
full       1808
ghost       437
partial      35
Name: count, dtype: int64

Referees in analysis sample: 33
Referees with at least one ghost match: 19

Missing values in key columns: 0


In [25]:
# How many ghost and crowd matches each referee had — the referee fixed effect compares these
referee_table = pd.crosstab(analysis["Referee"], analysis["crowd"])
referee_table.sort_values("ghost", ascending=False)

crowd,full,ghost,partial
Referee,,,
M Oliver,139,35,2
A Taylor,140,34,3
M Atkinson,108,31,2
A Marriner,104,29,3
C Pawson,112,28,3
J Moss,101,28,2
S Attwell,95,27,2
M Dean,102,27,3
K Friend,86,26,4


In [26]:
OUTPUT_COLUMNS = [
    "season", "season_start", "Date", "HomeTeam", "AwayTeam", "Referee",
    "FTHG", "FTAG", "HF", "AF", "HY", "AY", "HR", "AR",
    "crowd", "ghost", "partial", "attendance", "low_fouls",
    "card_diff", "cpf_diff", "foul_diff", "red_diff", "var", "trend",
]

PROCESSED_DIR.mkdir(exist_ok=True)
matches[OUTPUT_COLUMNS].to_csv(PROCESSED_DIR / "matches.csv", index=False)
print("Saved", matches.shape[0], "rows x", len(OUTPUT_COLUMNS), "columns to data/processed/matches.csv")

Saved 4940 rows x 25 columns to data/processed/matches.csv
